In [ ]:
!pip install -q transformers torch torchvision pillow matplotlib pandas

In [ ]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
IMAGE_PATH = "/kaggle/input/datasets/linataacha/kaggleinputlab/cat2.jpg"

image = Image.open(IMAGE_PATH).convert("RGB")
print("Image size:", image.size)
print("Image mode:", image.mode)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off");

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_PATH = "/kaggle/input/datasets/linataacha/kaggleinputlab/cat2.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")
print("Image size:", image.size)
print("Image mode:", image.mode)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off");

In [ ]:
texts = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a car",
    "a photo of a horse",
    "a photo of a person"
]

inputs = processor(
    text=texts,
    images=image,
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probs = outputs.logits_per_image.softmax(dim=1)[0].cpu()

for text, probability in zip(texts, probs):
    print(f"{text:35s}: {probability.item():.4f}")

In [ ]:
def classify_image(image, classes):
    prompts = [f"a photo of a {c}" for c in classes]

    inputs = processor(text=prompts, images=image,
                       return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1)[0].cpu()
    best_idx = probs.argmax().item()

    return classes[best_idx], probs[best_idx].item()   # fin de la fonction


classes = ["cat", "dog", "car", "airplane", "horse", "bird"]

prediction, probability = classify_image(image, classes)
print("Prediction:", prediction)
print("Probability:", probability)

In [ ]:
import os

paths = {}
for root, dirs, files in os.walk("/kaggle/input/datasets/linataacha/kaggleinputlab/"):
    for f in files:
        paths[f] = os.path.join(root, f)

print(list(paths.keys()))  

In [ ]:
print(len(paths))
for name, full in sorted(paths.items()):
    print(repr(name), "->", full)

In [ ]:
dataset = [
    ("cat2.jpg", "cat"),
    ("cat1.jpg", "cat"),
    ("dog1.jpg", "dog"),
    ("dog2.jpg", "dog"),
    ("car1.jpg", "car"),
    ("car2.jpg", "car"),
    ("bird1.jpg", "bird"),
    ("bird2.jpg", "bird"),
    ("house.jpg", "house"),
    ("house2.jpg", "house"),
    ("horse.jpg", "horse")
]

classes = ["cat", "dog", "car", "bird", "house"]

rows = []
for path, true_class in dataset:
    img = Image.open(paths[path]).convert("RGB")
    pred, prob = classify_image(img, classes)
    rows.append({
        "image": path,
        "actual": true_class,
        "predicted": pred,
        "probability": round(prob, 4),
        "correct": pred == true_class
    })

results = pd.DataFrame(rows)
print(results)

accuracy = results["correct"].mean()
print(f"Accuracy: {accuracy:.2%}  ({results['correct'].sum()}/{len(results)})")

print(results[~results["correct"]])

In [ ]:
dataset = [
    ("cat2.jpg", "cat"),
    ("cat1.jpg", "cat"),
    ("dog1.jpg", "dog"),
    ("dog2.jpg", "dog"),
    ("car1.jpg", "car"),
    ("car2.jpg", "car"),
    ("bird1.jpg", "bird"),
    ("bird2.jpg", "bird"),
    ("house.jpg", "house"),
    ("house2.jpg", "house")
]

classes = ["cat", "dog", "car", "bird", "house"]

rows = []
for path, true_class in dataset:
    img = Image.open(paths[path]).convert("RGB")
    pred, prob = classify_image(img, classes)
    rows.append({
        "image": path,
        "actual": true_class,
        "predicted": pred,
        "probability": round(prob, 4),
        "correct": pred == true_class
    })

results = pd.DataFrame(rows)
print(results)

accuracy = results["correct"].mean()
print(f"Accuracy: {accuracy:.2%}  ({results['correct'].sum()}/{len(results)})")

print(results[~results["correct"]])

In [ ]:
image = Image.open(paths["cat1.jpg"]).convert("RGB")

prompts = [
    "a cat",
    "a photo of a cat",
    "a photograph showing a cat",
    "an image containing a cat",
    "a domestic animal called a cat"
]

inputs = processor(text=prompts, images=image,
                   return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits_per_image[0].cpu()
cosine = logits / model.logit_scale.exp().item()

df = pd.DataFrame({
    "prompt": prompts,
    "logit": logits.numpy().round(3),
    "cosine": cosine.numpy().round(4)
}).sort_values("logit", ascending=False).reset_index(drop=True)

print(df)
print("\nMeilleur prompt :", df.loc[0, "prompt"])

In [ ]:

IMAGE_DIR = "/kaggle/input/datasets/linataacha/kaggleinputlab/"     
QUERY = "a person playing football"
EXTENSIONS = (".jpg")

image_paths = []
for root, dirs, files in os.walk(IMAGE_DIR):
    for f in sorted(files):
        if f.lower().endswith(EXTENSIONS):
            image_paths.append(os.path.join(root, f))

print(f"{len(image_paths)} images trouvées")  


def to_tensor(x):

    return x if torch.is_tensor(x) else x.pooler_output


def embed_images(paths, batch_size=8):
    all_embs = []
    for i in range(0, len(paths), batch_size):
        batch = [Image.open(p).convert("RGB") for p in paths[i:i + batch_size]]
        inputs = processor(images=batch, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            feats = to_tensor(model.get_image_features(**inputs))
        feats = feats / feats.norm(dim=-1, keepdim=True)   
        all_embs.append(feats.cpu())
    return torch.cat(all_embs)


image_embeddings = embed_images(image_paths)
print("Forme de la matrice d'embeddings :", image_embeddings.shape)   

In [ ]:
inputs = processor(text=[QUERY], return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    q = to_tensor(model.get_text_features(**inputs))
q = (q / q.norm(dim=-1, keepdim=True)).cpu()          

scores = (image_embeddings @ q.T).squeeze(1)           

top_scores, top_idx = scores.topk(5)

plt.figure(figsize=(16, 4))
for rank, (s, idx) in enumerate(zip(top_scores, top_idx), start=1):
    plt.subplot(1, 5, rank)
    plt.imshow(Image.open(image_paths[idx]).convert("RGB"))
    plt.title(f"#{rank}  cos={s:.3f}\n{os.path.basename(image_paths[idx])}", fontsize=9)
    plt.axis("off")
plt.suptitle(f'Requête : "{QUERY}"')
plt.show()

In [ ]:
!pip install -q bitsandbytes accelerate

import gc, torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor_llava = AutoProcessor.from_pretrained(MODEL_ID)
llava = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("LLaVA chargé sur :", llava.device)

In [ ]:
from PIL import Image

def ask_llava(image, question, max_new_tokens=200):
    prompt = f"USER: <image>\n{question} ASSISTANT:"

    inputs = processor_llava(images=image, text=prompt, return_tensors="pt")
    inputs = inputs.to(llava.device, torch.float16)  
    with torch.no_grad():
        output = llava.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return processor_llava.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
img = Image.open(paths["soccer2.jpg"]).convert("RGB")  

questions = [
    "What is happening in this image?",
    "How many people are visible?",
    "What objects can you identify?",
    "Describe the image in detail.",
]

answers = {}
for q in questions:
    answers[q] = ask_llava(img, q, max_new_tokens=300 if "detail" in q else 150)
    print(f"Q : {q}\nR : {answers[q]}\n{'-'*60}")

In [ ]:
import pandas as pd
pd.DataFrame({"question": list(answers.keys()), "réponse LLaVA": list(answers.values())})

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

plan = {
    "dog2.jpg": [
        ("comptage", "How many people/animal are visible?"),
        ("objets",   "What objects can you see?"),
        ("lieu",     "Where is the animal?"),
        ("action",   "What is the animal doing?"),
        ("couleur",  "What color is the color of the object?"),
    ],
    "f1.jpg": [
        ("comptage", "How many people are visible?"),
        ("objets",   "What objects can you see?"),
        ("lieu",     "Where are the people?"),
        ("action",   "What are they doing?"),
        ("texte",    "What is written on the sign?"),
    ],
    "horsepower1.jpg": [
        ("comptage", "How many things are visible?"),
        ("objets", "What objects can you see?"),
        ("lieu", "Where these things can be used?"),
        ("action", "What are they doing?"),
        ("relation", "What is the relation between these objects?")
    ],
    "f3.jpg": [
        ("comptage", "How many people are visible?"),
        ("objets", "What objects can you see?"),
        ("lieu", "Where these things can be used?"),
        ("action", "What are they doing?"),
        ("couleur", "What color are they?")
    ],
    "horsepwer2.jpg": [
        ("relation", "What is on the left of the car?"),
        ("relation", "What is connecting the horse to the car?"),
        ("couleur", "What color is the car?"),
        ("comptage", "How many wheels of the car are visible?"),
        ("piege", "What is written on the car's license plate?")
    ]
}

rows = []
for name, qs in plan.items():
    img = Image.open(paths[name]).convert("RGB")
    for qtype, q in qs:
        answer = ask_llava(img, q + " Answer briefly.", max_new_tokens=80)
        rows.append({
            "Image": name,
            "Type": qtype,
            "Question": q,
            "LLaVA Answer": answer,
            "Human Evaluation": None,     
        })
        print(f"{name} | {q}\n -> {answer}\n")

vqa = pd.DataFrame(rows)
vqa.to_csv("vqa_results.csv", index=False)   

In [ ]:
vqa["Human Evaluation"] = [
    "correct", "correct", "partiel", "faux", "correct",
    "correct", "correct", "correct", "correct", "faux",
    "correct", "correct", "correct", "correct", "partiel",
    "correct", "correct", "correct", "correct", "faux",
    "correct", "correct", "faux", "faux", "correct"
]

In [ ]:
n_total = len(vqa)
n_correct = (vqa["Human Evaluation"] == "correct").sum()
n_partial = (vqa["Human Evaluation"] == "partiel").sum()

acc_strict = n_correct / n_total
acc_souple = (n_correct + 0.5 * n_partial) / n_total

print(f"Accuracy stricte : {n_correct}/{n_total} = {acc_strict:.2%}")
print(f"Accuracy avec crédit partiel (0,5) : {acc_souple:.2%}")

by_type = (vqa.assign(ok=vqa["Human Evaluation"] == "correct")
              .groupby("Type")["ok"].agg(["mean", "count"])
              .sort_values("mean"))
print(by_type)

In [ ]:
img = Image.open(paths["books.jpg"]).convert("RGB")   
plt.imshow(img); plt.axis("off"); plt.show()        

questions = [
    "Is there a bicycle in this image?",
    "What color is the bicycle?",
    "Where is the bicycle located?",
]

halluc_rows = []
for q in questions:
    answer = ask_llava(img, q, max_new_tokens=100)
    halluc_rows.append({"Question": q, "LLaVA Answer": answer})
    print(f"Q : {q}\nR : {answer}\n{'-'*60}")

pd.DataFrame(halluc_rows)

In [ ]:
for q in ["Is there a book in this image?",
          "What color is the book?",
          "Where is the book located?"]:
    print(q, "->", ask_llava(img, q, max_new_tokens=100), "\n")

In [ ]:
q = "If there is a bicycle in this image, what color is it? If there is none, say so."
print(ask_llava(img, q, max_new_tokens=100))

In [ ]:
import time

img = Image.open(paths["horse.jpg"]).convert("RGB")
plt.imshow(img); plt.axis("off"); plt.show()

# ---------- CLIP ----------
texts = ["a photo of a cat", "a photo of a dog", "a photo of a horse"]

t0 = time.time()
inputs = processor(text=texts, images=img, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    out = model(**inputs)
probs = out.logits_per_image.softmax(dim=1)[0].cpu()
t_clip = time.time() - t0

print("=== CLIP ===")
for t, p in zip(texts, probs):
    print(f"{t:25s}: {p.item():.4f}")
print(f"Temps : {t_clip:.2f} s\n")

# ---------- LLaVA ----------
question = "What animal is present in the image? Describe its appearance."

t0 = time.time()
answer = ask_llava(img, question, max_new_tokens=200)
t_llava = time.time() - t0

print("=== LLaVA ===")
print(answer)
print(f"Temps : {t_llava:.2f} s")

In [ ]:
import time
import matplotlib.pyplot as plt
from PIL import Image

def multimodal_assistant(image, question, max_new_tokens=150, show=True):
    """Image (chemin ou objet PIL) + question -> réponse de LLaVA."""
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")

    t0 = time.time()
    answer = ask_llava(image, question, max_new_tokens=max_new_tokens)
    dt = time.time() - t0

    if show:
        plt.figure(figsize=(5, 5))
        plt.imshow(image); plt.axis("off")
        plt.title(question, fontsize=10)
        plt.show()
        print(f"Réponse : {answer}\n(temps : {dt:.2f} s)")
    return answer

# Démonstrations
multimodal_assistant(paths["horse.jpg"], "What animal is present in the image?")
multimodal_assistant(paths["soccer2.jpg"], "What is the person doing?")

In [ ]:
def retrieve_and_explain(query, k=3):
    # 1) CLIP : embedding de la requête, similarité, top-k
    inputs = processor(text=[query], return_tensors="pt", padding=True)
    inputs = {key: v.to(device) for key, v in inputs.items()}
    with torch.no_grad():
        q = to_tensor(model.get_text_features(**inputs))
    q = (q / q.norm(dim=-1, keepdim=True)).cpu()

    scores = (image_embeddings @ q.T).squeeze(1)
    top_scores, top_idx = scores.topk(k)

    # 2) LLaVA : explication pour chaque image retenue
    prompt = (f'The user searched for: "{query}". '
              "If this image matches the search, explain briefly why. "
              "If it does not match, say so.")

    for rank, (s, idx) in enumerate(zip(top_scores, top_idx), start=1):
        img = Image.open(image_paths[idx]).convert("RGB")
        explanation = ask_llava(img, prompt, max_new_tokens=120)

        plt.figure(figsize=(4, 4))
        plt.imshow(img); plt.axis("off")
        plt.title(f"#{rank}  cos={s:.3f}  {os.path.basename(image_paths[idx])}", fontsize=9)
        plt.show()
        print(f"LLaVA : {explanation}\n{'-'*60}")

retrieve_and_explain("a person playing football", k=3)

Reponse des questions :
Q1: Contrastive Language-Image Pre-training (OpenAI, 2021) est un modèle qui projette images et textes dans un même espace vectoriel. Il est entraîné par apprentissage contrastif sur environ 400 millions de paires image-légende : les vraies paires sont rapprochées, les fausses éloignées. Cela permet la classification zero-shot et la recherche d’images sans entraînement supplémentaire.
Q2: Un encodeur d’image (ici un Vision Transformer ViT-B/32) et un encodeur de texte (un Transformer). Chacun est suivi d’une projection linéaire vers la même dimension (512), ce qui rend les vecteurs comparables par similarité cosinus.
Q3: Il prépare les données pour le modèle. Pour l’image : redimensionnement à 224×224, recadrage, conversion en tenseur et normalisation. Pour le texte : tokenisation, padding et troncature à 77 tokens. Il produit des tenseurs PyTorch.
Q4: C’est le réseau de neurones lui-même (les deux encodeurs, les projections et le paramètre de température). Il calcule les embeddings et les scores de similarité image-texte (logits_per_image, logits_per_text).
Q5: Image size: (700, 798),soit largeur × hauteur en pixels. CLIP la redimensionne ensuite à 224×224.
Q6: Une grille de pixels. En mode RGB, chaque pixel est un triplet d’entiers de 0 à 255 (uint8). L’objet porte aussi des métadonnées (taille, mode, format). Ce n’est pas encore un tenseur : la conversion en tenseur flottant normalisé est faite par CLIPProcessor.
Q7: a photo of a cat  : 0.9841
Q8: Oui, la phrase correspond a ce que je vois dans l'image. CLIP a alors correctement aligné l’image et le texte.
Q9: Parce que l’entraînement contrastif a forcé les deux encodeurs à projeter dans le même espace vectoriel de 512 dimensions. Une image et sa description y sont proches, des paires non correspondantes sont éloignées. Comparer revient à mesurer une similarité cosinus entre deux vecteurs du même espace.
Q10: Elle transforme un vecteur de scores quelconques en distribution de probabilités : chaque valeur est exponentiée puis divisée par la somme des exponentielles, $p_i = e^{z_i} / \sum_j e^{z_j}$. Les valeurs sont entre 0 et 1, leur somme vaut 1, et l’ordre est conservé. Ces probabilités sont relatives aux candidats proposés, pas une certitude absolue.
Q11: Classer des images dans des catégories pour lesquelles le modèle n’a reçu aucun exemple étiqueté pour la tâche, en décrivant simplement ces catégories par du langage.
Q12: Parce que le classifieur est construit à la volée par l’encodeur de texte : chaque nom de classe devient un vecteur qui joue le rôle de poids de classe. L’alignement image-texte a déjà été appris pendant le pré-entraînement. Changer de classes revient à changer la liste de textes.
Q13: Le softmax répartit 100 % de la probabilité entre les candidats, donc CLIP choisit la classe la moins mauvaise, parfois avec une forte probabilité. Dans ton test, le cheval, sans horse dans la liste, a été classé dog (0,752).
Q14: Sur les 10 images normales : 10/10 = 100 %. En ajoutant volontairement une image de cheval dont la classe est absente de la liste : 10/11 = 90,91 %. Précise que les classes candidates étaient cat, dog, car, bird, house.
Q15: Une seule : horse.jpg, classe réelle horse, prédite dog, probabilité 0,752. Les 10 autres sont correctes (probabilités de 0,9917 à 0,9996).
Q16: La classe horse était absente des candidats : CLIP ne pouvait pas la prédire, et il a choisi le quadrupède le plus proche (dog).
Proximité visuelle entre un cheval et un chien (quatre pattes, corps d’animal).
Pour le reste du test : images faciles, peu de classes, échantillon de 10 images seulement, donc une accuracy de 100 % qui ne prouve pas la robustesse. Les erreurs possibles sur des images ambiguës, floues, avec plusieurs objets ou des classes proches n’ont pas été testées.
Q17: Non. Les logits vont de 26,05 à 28,10, soit un écart d’environ 2 points, bien que les 5 phrases désignent le même chat. Chaque formulation produit un vecteur différent.
Q18: "a photo of a cat" (logit 28,10 ; cosinus 0,281), suivi de "an image containing a cat" (27,51), "a photograph showing a cat" (27,42), "a cat" (26,33) et "a domestic animal called a cat" (26,05).
Q19: CLIP est sensible à la formulation. Les prompts qui ressemblent à une légende d’image (« a photo of... ») battent le mot nu, et une phrase plus savante (« a domestic animal called a cat ») arrive dernière : plus de mots ne veut pas dire mieux. Le prompt est un hyperparamètre à tester, et on peut moyenner plusieurs prompts (prompt ensembling). Ce résultat porte sur une seule image.
Q20: soccer1.jpg, avec une similarité cosinus de 0,293. Top-5 : soccer1 (0,293), soccer2 (0,291), football1 (0,275), dog2 (0,228), f1 (0,214).
Q21: Oui : elle montre un joueur de football et un ballon sur une pelouse. Les rangs 1 à 3 sont pertinents (le rang 3 est du football américain, proche sémantiquement). Les rangs 4 (chien avec une balle) et 5 (pilote de F1) sont des faux positifs. Le score chute de 0,275 à 0,228 entre les rangs 3 et 4, ce qui suggère qu’un seuil de similarité aiderait à ne garder que les bons résultats.
Q22: Parce que chaque image est représentée par son embedding visuel, qui encode déjà son contenu, et que la requête en langage naturel est projetée dans le même espace. On mesure simplement la similarité entre les deux : aucun tag ni catégorie n’a besoin d’être associé manuellement aux images.
Q23: La recherche dans une photothèque (Google Photos, Apple Photos) : taper « plage au coucher du soleil » pour retrouver des photos jamais étiquetées. Le bénéfice est l’absence d’annotation manuelle et la possibilité de requêtes en langage naturel. L’e-commerce en est un autre exemple.
Q24: Schéma complété : Image → CLIP Vision Encoder → Projection Layer → Language Model → Text Response.
Q25: Transformer l’image en représentation numérique riche : un vecteur par patch, qui encode le contenu visuel. Il joue le rôle des yeux du système. Utiliser CLIP aide, car ses représentations sont déjà alignées avec le langage.
Q25: L’encodeur de vision et le LLM ont été entraînés séparément, donc leurs espaces vectoriels diffèrent (en dimension et en signification). La projection traduit les vecteurs visuels vers l’espace des embeddings de mots du LLM, qui les traite alors comme des tokens visuels. C’est aussi la partie centrale apprise pour relier les deux modèles.
Q26: Il reçoit les tokens visuels et la question, raisonne dessus avec ses connaissances et génère la réponse textuelle, token par token. C’est lui qui permet de décrire, d’expliquer et de répondre librement.
Q27: 18/25 = 72 % en évaluation stricte, 76 % avec crédit partiel (0,5 par réponse partielle). Le jugement est humain, donc subjectif, et sur 25 questions une réponse change le résultat de 4 points.
Q28: Type	Accuracy	Nb
     texte	0 %	1
     couleur	33 %	3
     relation	67 %	3
     action	75 %	4
     lieu	75 %	4
     comptage	80 %	5
     objets	100 %	4
     piège	100 %	1
Les questions de reconnaissance d’objets et de description de la scène sont bien traitées. La couleur est la catégorie la plus faible. Le comptage est bon globalement mais a produit une erreur révélatrice : « 4 roues » au lieu de 2 visibles, par a priori. La lecture de texte semble difficile (résolution de 336×336), mais avec une seule question, c’est un indice et non une conclusion.
Q29: Le vélo existe-t-il réellement ?
Non. L’image montre une pile de livres sur une table en bois.
Q30: Oui. Il a d’abord répondu correctement « No, there is no bicycle », puis a inventé une couleur (« blue ») et une position (« on a wooden table »). Quand la question autorise une réponse négative (« If there is none, say so »), il répond correctement qu’il n’y a pas de vélo. Ce résultat porte sur une seule image.
Q31: Une réponse d’un modèle multimodal qui décrit ou affirme quelque chose d’absent de l’image (objet, attribut ou relation inventés), formulée avec assurance et fluidité, ce qui la rend crédible.
Q32: Il suit la fausse prémisse de la question plutôt que de la contredire.
Le LLM génère le texte le plus probable selon ses habitudes de langage, plus que selon l’image.
La génération est auto-régressive : chaque mot produit devient du contexte pour le suivant.
L’information visuelle est limitée (576 tokens à 336×336), ce qui laisse le LLM combler les vides.
L’alignement vision-langage est imparfait, et les données d’ajustement peuvent favoriser les réponses affirmatives.
Q33: CLIP : il a classé le cheval à 99,86 % en 0,03 s, avec une probabilité exploitable et reproductible. LLaVA a aussi trouvé « horse », mais en texte libre, sans score, environ 65 fois plus lentement (1,98 s).
Q34: LLaVA, car CLIP ne génère aucun texte. 
Q35: LLaVA, seul des deux à accepter une question ouverte et à produire une réponse. CLIP ne peut que comparer des phrases proposées.
Q36: CLIP apprend un espace commun image-texte et mesure la correspondance entre une image et un texte, sans rien générer. LLaVA réutilise un encodeur de type CLIP, y ajoute une projection et un LLM génératif, et produit du texte. CLIP évalue, LLaVA génère. Le premier est rapide, mesurable et fiable pour classer et chercher, mais limité aux candidats fournis. Le second est flexible, mais plus lent et sujet aux hallucinations.
